## Константы для всех рассчетов (Шаг 4)

In [1]:
# ШАГ 4 плана До 30.10. Константы. Остальные ячейки берут числа только отсюда.
import numpy as np

SEED = 42
np.random.seed(SEED)

MU = np.array([30.0, 42.0, 45.0, 48.0, 52.0]) # центры x0..x4
SIGMA = np.array([12.0, 15.0, 16.0, 18.0, 20.0]) # ширины x0..x4
U_MU = 0.02  # центр u
U_SIGMA = 0.015 # ширина u
COS_AMP = 0.15  # амплитуда косинусного множителя
COS_K = 0.04    # cos(0.04 * (x1 + x2 - 2*x0))
C_COS = COS_K * np.array([-2.0, 1.0, 1.0, 0.0, 0.0]) # коэффициенты при x0..x4 внутри косинуса: -0.08, 0.04, 0.04, 0, 0

A_BOUNDS = np.array([0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # область D = [0,100]^5 x [0,0.08]
B_BOUNDS = np.array([100.0, 100.0, 100.0, 100.0, 100.0, 0.08])
vol_D = float(np.prod(B_BOUNDS - A_BOUNDS)) # объём D = 100^5 * 0.08

L_EXC = 250          # L из задания (избыточный запас)
N_POINTS = 100_000   # размер выборки

print("SEED =", SEED, "  vol_D =", vol_D)

SEED = 42   vol_D = 800000000.0


## Функция g, точный I(D), выборка и обучение сети с той же структурой

In [2]:
# Подготовка 1. Функция g (формула (3)) и точный I(D)
from scipy.integrate import quad

def g_function(xi):
    """Формула (3) задания. xi: (N, 6)."""
    xi = np.atleast_2d(xi)
    exp_sum = (((xi[:, :5] - MU) ** 2)/(2.0 * SIGMA ** 2)).sum(axis=1)
    exp_sum += (xi[:, 5] - U_MU) ** 2/(2.0 * U_SIGMA ** 2)
    cos_part = 1.0 + COS_AMP * np.cos(COS_K * (xi[:, 1] + xi[:, 2] - 2.0 * xi[:, 0]))
    return np.exp(-exp_sum) * cos_part

A_k, B_k = [], []
for k in range(5):
    G = lambda x, k=k: np.exp(-(x - MU[k]) ** 2 / (2 * SIGMA[k] ** 2))
    A_k.append(quad(G, A_BOUNDS[k], B_BOUNDS[k])[0])
    re = quad(lambda x, k=k: G(x) * np.cos(C_COS[k] * x), A_BOUNDS[k], B_BOUNDS[k])[0]
    im = quad(lambda x, k=k: G(x) * np.sin(C_COS[k] * x), A_BOUNDS[k], B_BOUNDS[k])[0]
    B_k.append(re + 1j * im)
Au = quad(lambda u: np.exp(-(u - U_MU) ** 2 / (2 * U_SIGMA ** 2)), A_BOUNDS[5], B_BOUNDS[5])[0]
I_D = Au * (np.prod(A_k) + COS_AMP * np.real(np.prod(B_k)))
print(f"Точный I(D) = {I_D:.4f}   (у преподавателя 3 506 303)")

Точный I(D) = 3506302.9794   (у преподавателя 3 506 303)


In [3]:
# Выборка и "старая" сеть (100 000 равномерных точек, 50 нейронов, 150 эпох)
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input
tf.random.set_seed(SEED)

X_raw = np.random.uniform(A_BOUNDS, B_BOUNDS, (N_POINTS, 6))
y_raw = g_function(X_raw)
X_scaled = (X_raw - A_BOUNDS) / (B_BOUNDS - A_BOUNDS)  # в [0, 1]
y_min, y_max = y_raw.min(), y_raw.max()
y_scaled = (y_raw - y_min) / (y_max - y_min)
split = int(0.8 * N_POINTS)     # 80/20

inputs = Input(shape=(6,))
hidden = Dense(50, activation='sigmoid', name='hidden')(inputs)
outputs = Dense(1, activation='linear', name='output')(hidden)
model = Model(inputs=inputs, outputs=outputs)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.01), loss='mse')
history = model.fit(X_scaled[:split], y_scaled[:split], validation_data=(X_scaled[split:], y_scaled[split:]),
          epochs=150, batch_size=256, verbose=0)

# Предсказания сети на всей выборке, в исходном масштабе g, и координаты точек
y_pred = model.predict(X_scaled, batch_size=4096, verbose=0).flatten() * (y_max - y_min) + y_min
x0, x1, x2, x3, x4, u = [X_raw[:, i] for i in range(6)]
print("y_pred:", y_pred.shape)

y_pred: (100000,)


In [4]:
print("======= ОТЧЕТ ОБ ОБУЧЕНИИ СЕТИ =======")
print(f"Форма массива предсказаний y_pred: {y_pred.shape}")
print(f"Финальный Loss на обучении (MSE):   {history.history['loss'][-1]:.6f}")
print(f"Финальный Loss на валидации (MSE): {history.history['val_loss'][-1]:.6f}")
print("---------------------------------------")
print("Сравнение истинной g и предсказаний сети (первые 5 точек):")
for i in range(5):
    print(f"  Точка {i}: Истинная g = {y_raw[i]:.4f} | Сеть y_pred = {y_pred[i]:.4f}")
print("=======================================")

======= ОТЧЕТ ОБ ОБУЧЕНИИ СЕТИ =======
Форма массива предсказаний y_pred: (100000,)
Финальный Loss на обучении (MSE):   0.000597
Финальный Loss на валидации (MSE): 0.000601
---------------------------------------
Сравнение истинной g и предсказаний сети (первые 5 точек):
  Точка 0: Истинная g = 0.0002 | Сеть y_pred = -0.0053
  Точка 1: Истинная g = 0.0000 | Сеть y_pred = -0.0000
  Точка 2: Истинная g = 0.0134 | Сеть y_pred = 0.0227
  Точка 3: Истинная g = 0.0049 | Сеть y_pred = 0.0129
  Точка 4: Истинная g = 0.0000 | Сеть y_pred = -0.0087


## Шаг 1: Исправлен R_j = x0 + x1 + … + xj (j+1 слагаемых), в обеих функциях

In [5]:
# ШАГ 1 плана До 30.10. Две функции. Изменён только расчёт R_j.

def compute_probabilities_fast(T_idx, s, Q):
    """NNI: значения сети y_pred в общих точках + маски областей."""
    T_values = [7, 14, 21, 28]
    T = T_values[T_idx]
    j = T_idx + 1

    # R_j = x0 + x1 + ... + xj  (j+1 слагаемых; для j=4 это x0..x4)
    if j == 1:
        R_j = x0 + x1
    elif j == 2:
        R_j = x0 + x1 + x2
    elif j == 3:
        R_j = x0 + x1 + x2 + x3
    else:
        R_j = x0 + x1 + x2 + x3 + x4

    A_Q = Q * (1 - u)
    mask_Omega0 = R_j <= s
    mask_Omega2 = R_j > s + A_Q
    mask_Omega3 = (s + A_Q - R_j) > L_EXC

    I0 = vol_D * np.mean(y_pred * mask_Omega0)
    I2 = vol_D * np.mean(y_pred * mask_Omega2)
    I3 = vol_D * np.mean(y_pred * mask_Omega3)

    P0, P2, P3 = I0 / I_D, I2 / I_D, I3 / I_D
    P1 = 1 - P0 - P2    # формула (14)
    return {'P0': P0, 'P1': P1, 'P2': P2, 'P3': P3, 'sum': P0 + P1 + P2}


def compute_probabilities_pure_monte_carlo(T_idx, s, Q, N_mc=100000):
    """Монте-Карло по настоящей g, без нейросети."""
    T_values = [7, 14, 21, 28]
    T = T_values[T_idx]
    j = T_idx + 1

    X_mc = np.random.uniform(A_BOUNDS, B_BOUNDS, (N_mc, 6))
    y_true = g_function(X_mc)
    x0_mc, x1_mc, x2_mc, x3_mc, x4_mc = X_mc[:, 0], X_mc[:, 1], X_mc[:, 2], X_mc[:, 3], X_mc[:, 4]
    u_mc = X_mc[:, 5]

    # R_j = x0 + x1 + ... + xj  (j+1 слагаемых; для j=4 это x0..x4)
    if j == 1: R_j = x0_mc + x1_mc
    elif j == 2: R_j = x0_mc + x1_mc + x2_mc
    elif j == 3: R_j = x0_mc + x1_mc + x2_mc + x3_mc
    else: R_j = x0_mc + x1_mc + x2_mc + x3_mc + x4_mc

    A_Q = Q * (1 - u_mc)
    m0 = R_j <= s
    m2 = R_j > s + A_Q
    m3 = (s + A_Q - R_j) > L_EXC

    P0 = vol_D * np.mean(y_true * m0) / I_D
    P2 = vol_D * np.mean(y_true * m2) / I_D
    P3 = vol_D * np.mean(y_true * m3) / I_D
    P1 = 1 - P0 - P2
    return {'P0': P0, 'P1': P1, 'P2': P2, 'P3': P3}

In [6]:
# ПРОВЕРКА: для точки (1, 2, 3, 4, 5, 0.01) должно получиться R1=3, R2=6, R3=10, R4=15
# Подставляем одну тестовую точку,плотность = 1, vol_D = I_D = 1. Тогда P0 = 1, если R_j <= s, иначе 0.
# Значит R_j — порог s, на котором P0 переключается.
print("Проверка исправления R_j:")
pt = np.array([1.0, 2.0, 3.0, 4.0, 5.0, 0.01])
expected_R = [3, 6, 10, 15]

saved = dict(x0=x0, x1=x1, x2=x2, x3=x3, x4=x4, u=u, y_pred=y_pred, vol_D=vol_D, I_D=I_D,
             g_function=g_function, uniform=np.random.uniform)
try:
    x0, x1, x2, x3, x4, u = [np.array([v]) for v in pt]
    y_pred = np.array([1.0]); vol_D = 1.0; I_D = 1.0
    np.random.uniform = lambda lo, hi, size: np.tile(pt, (size[0], 1))
    g_function = lambda xi: np.ones(len(np.atleast_2d(xi)))

    ok = True
    print(f"{'j':<3}{'ожидается R_j':>15}{'NNI':>8}{'Монте-Карло':>14}")
    for j, R_exp in enumerate(expected_R, start=1):
        res = []
        for fn in (lambda s: compute_probabilities_fast(T_idx=j - 1, s=s, Q=100)['P0'],
                   lambda s: compute_probabilities_pure_monte_carlo(T_idx=j - 1, s=s, Q=100, N_mc=1)['P0']):
            res.append(fn(R_exp - 0.5) == 0.0 and fn(R_exp) == 1.0)
        ok &= all(res)
        print(f"{j:<3}{R_exp:>15}{str(res[0]):>8}{str(res[1]):>14}")
finally:
    x0, x1, x2, x3, x4, u = saved['x0'], saved['x1'], saved['x2'], saved['x3'], saved['x4'], saved['u']
    y_pred, vol_D, I_D = saved['y_pred'], saved['vol_D'], saved['I_D']
    g_function = saved['g_function']; np.random.uniform = saved['uniform']

Проверка исправления R_j:
j    ожидается R_j     NNI   Монте-Карло
1                3    True          True
2                6    True          True
3               10    True          True
4               15    True          True


## Шаг 2. Пересчет трех контрольных политик
P0, P1, P2 по NNI (маски по значениям сети) и по Монте-Карло. Это контрольные числа после исправления.

In [8]:
# ШАГ 2
control_policies = [(14, 100, 300), (21, 150, 200), (28, 100, 100)]  # (T, s, Q)
T_to_idx = {7: 0, 14: 1, 21: 2, 28: 3}

journal_step2 = []
print(f"{'политика (T, s, Q)':<20}{'метод':<14}{'P0':>10}{'P1':>10}{'P2':>10}")
for (T, s, Q) in control_policies:
    r_nni = compute_probabilities_fast(T_idx=T_to_idx[T], s=s, Q=Q)
    r_mc  = compute_probabilities_pure_monte_carlo(T_idx=T_to_idx[T], s=s, Q=Q, N_mc=200000)
    for name, r in (("NNI", r_nni), ("Монте-Карло", r_mc)):
        print(f"{str((T, s, Q)):<20}{name:<14}{r['P0']:10.4f}{r['P1']:10.4f}{r['P2']:10.4f}")
        journal_step2.append(((T, s, Q), name, r['P0'], r['P1'], r['P2']))

политика (T, s, Q)  метод                 P0        P1        P2
(14, 100, 300)      NNI               0.3134    0.6866    0.0000
(14, 100, 300)      Монте-Карло       0.2471    0.7529    0.0000
(21, 150, 200)      NNI               0.3709    0.6317   -0.0027
(21, 150, 200)      Монте-Карло       0.3087    0.6913    0.0000
(28, 100, 100)      NNI              -0.0064    0.0882    0.9181
(28, 100, 100)      Монте-Карло       0.0004    0.2914    0.7082


Видно отрицательные вероятности и расхождения, но пока они могут остаться.

## Шаг 3. Git

Тег на **старую** версию v0-before-nni-fix

## Шаг 5. Ручная проверка g в трёх точках

In [ ]:
# Ручная проверка g в трёх точках
# Считайте на калькуляторе по формуле:
# g = exp( -sum_i (x_i-mu_i)^2 / (2 sigma_i^2) - (u-0.02)^2 / (2*0.015^2) ) * (1 + 0.15*cos(0.04*(x1+x2-2*x0)))
check_points = {
    # A: центр пика, все квадраты = 0, exp(0) = 1; cos(0.04*(42+45-2*30)) = cos(1.08)
    "A: центр (30,42,45,48,52,0.02)": (
        np.array([30.0, 42.0, 45.0, 48.0, 52.0, 0.02]),
        1.0 + 0.15 * np.cos(1.08)
    ),

    # B: точка далеко от центра
    # exp sum = 400/288 + 484/450 + 225/512 + 484/648 + 784/800 + 0.0016/0.00045 = 8.1864
    # cos(0.04 * (20 + 30 - 2*10)) = cos(1.2)
    "B: угловая (10,20,30,70,80,0.06)": (
        np.array([10.0, 20.0, 30.0, 70.0, 80.0, 0.06]),
        np.exp(-(400/288 + 484/450 + 225/512 + 484/648 + 784/800 + 0.0016/0.00045)) * (1.0 + 0.15 * np.cos(1.2))
    ),

    # C: произвольная точка в пространстве.
    # exp sum = 25/288 + 4/450 + 25/512 + 9/648 + 9/800 + 0.0001/0.00045 = 0.3918841
    # cos(0.04 * (40 + 50 - 2*25)) = cos(1.6)
    "C: произв. (25,40,50,45,55,0.01)": (
        np.array([25.0, 40.0, 50.0, 45.0, 55.0, 0.01]),
        np.exp(-(25/288 + 4/450 + 25/512 + 9/648 + 9/800 + 0.0001/0.00045)) * (1.0 + 0.15 * np.cos(1.6))
    ),
}

print(f"{'точка':<36}{'вручную':>12}{'код':>12}  совпало")
for name, (pt_, by_hand) in check_points.items():
    val = g_function(pt_)[0]
    print(f"{name:<36}{by_hand:12.6f}{val:12.6f}  {np.isclose(val, by_hand, rtol=1e-12)}")
    assert np.isclose(val, by_hand, rtol=1e-12)



точка                                    вручную         код  совпало
A: центр (30,42,45,48,52,0.02)          1.070699    1.070699  True
B: угловая (10,20,30,70,80,0.06)        0.000294    0.000294  True
C: произв. (25,40,50,45,55,0.01)        0.672823    0.672823  True
